In [1]:
import os
os.chdir("/Users/karanrnaik/Desktop/earnings_analyzer")
import pandas as pd

mgmt_df = pd.read_csv("data/management_sentiment.csv")
topic_df = pd.read_csv("data/topic_assignments.csv")
print("✅ Data loaded")

✅ Data loaded


In [2]:
# Manually label 20 sentences to compare against FinBERT
# These are ground truth labels you assign by reading the text

sample = mgmt_df.sample(20, random_state=42)[['utterance','sentiment']].copy()
sample['ground_truth'] = [
    'neutral', 'positive', 'neutral', 'neutral', 'positive',
    'neutral', 'negative', 'neutral', 'positive', 'neutral',
    'neutral', 'positive', 'neutral', 'neutral', 'negative',
    'neutral', 'positive', 'neutral', 'neutral', 'positive'
]

from sklearn.metrics import classification_report, confusion_matrix

print("=== FinBERT Evaluation ===")
print(classification_report(
    sample['ground_truth'],
    sample['sentiment'],
    labels=['positive', 'negative', 'neutral']
))

=== FinBERT Evaluation ===


ValueError: Classification metrics can't handle a mix of multiclass and unknown targets

In [3]:
import pandas as pd

mgmt_df = pd.read_csv("data/management_sentiment.csv")

# First see what the sample actually looks like
sample = mgmt_df.sample(20, random_state=42)[['utterance','sentiment']].copy()
sample = sample.dropna(subset=['sentiment'])  # drop any NaN sentiment rows
sample = sample.reset_index(drop=True)

print("Sampled utterances and FinBERT predictions:")
for i, row in sample.iterrows():
    print(f"{i}: [{row['sentiment']}] {str(row['utterance'])[:80]}")

Sampled utterances and FinBERT predictions:
0: [neutral] Hi. Thanks. Krithi, just firstly on the next few quarters, can we expect the
1: [neutral] Thank you, Sashi. We go straight to Q&A; we can open it up and proceed. Neerav, 
2: [positive] Good evening, friends. First, let me wish all of you Shubha Dhanteras and Shubha
3: [positive] That's helpful. Thank you so much and all the best.
4: [neutral] Yes. Firstly, I want to say we have delivered our highest ever UVG in the last 1
5: [neutral] So, our guided range has been that 22% to 23%. We also outlined that the range,


In [4]:
for i, row in sample.iterrows():
    print(f"\n{i}: FinBERT → [{row['sentiment']}]")
    print(f"   Text: {str(row['utterance'])[:150]}")


0: FinBERT → [neutral]
   Text: Hi. Thanks. Krithi, just firstly on the next few quarters, can we expect the

1: FinBERT → [neutral]
   Text: Thank you, Sashi. We go straight to Q&A; we can open it up and proceed. Neerav, please open

2: FinBERT → [positive]
   Text: Good evening, friends. First, let me wish all of you Shubha Dhanteras and Shubha Deepavali.

3: FinBERT → [positive]
   Text: That's helpful. Thank you so much and all the best.

4: FinBERT → [neutral]
   Text: Yes. Firstly, I want to say we have delivered our highest ever UVG in the last 12

5: FinBERT → [neutral]
   Text: So, our guided range has been that 22% to 23%. We also outlined that the range,


In [5]:
print("ALL 20 ROWS — copy and share this output:\n")
for i, row in sample.iterrows():
    text = str(row['utterance'])[:120].replace('\n', ' ')
    print(f"{i:>2}: [{row['sentiment']:>8}] {text}")

ALL 20 ROWS — copy and share this output:

 0: [ neutral] Hi. Thanks. Krithi, just firstly on the next few quarters, can we expect the
 1: [ neutral] Thank you, Sashi. We go straight to Q&A; we can open it up and proceed. Neerav, please open
 2: [positive] Good evening, friends. First, let me wish all of you Shubha Dhanteras and Shubha Deepavali.
 3: [positive] That's helpful. Thank you so much and all the best.
 4: [ neutral] Yes. Firstly, I want to say we have delivered our highest ever UVG in the last 12
 5: [ neutral] So, our guided range has been that 22% to 23%. We also outlined that the range,


In [6]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

print(f"Total rows in sample: {len(sample)}\n")

for i in range(len(sample)):
    row = sample.iloc[i]
    text = str(row['utterance'])[:120].replace('\n', ' ')
    print(f"{i:>2}: [{row['sentiment']}] {text}")
    print()

Total rows in sample: 6

 0: [neutral] Hi. Thanks. Krithi, just firstly on the next few quarters, can we expect the

 1: [neutral] Thank you, Sashi. We go straight to Q&A; we can open it up and proceed. Neerav, please open

 2: [positive] Good evening, friends. First, let me wish all of you Shubha Dhanteras and Shubha Deepavali.

 3: [positive] That's helpful. Thank you so much and all the best.

 4: [neutral] Yes. Firstly, I want to say we have delivered our highest ever UVG in the last 12

 5: [neutral] So, our guided range has been that 22% to 23%. We also outlined that the range,



In [7]:
with open("data/sample_review.txt", "w") as f:
    for i in range(len(sample)):
        row = sample.iloc[i]
        f.write(f"{i}: [{row['sentiment']}]\n")
        f.write(f"   {str(row['utterance'])}\n\n")

print("✅ Saved to data/sample_review.txt — open this file in Finder")

✅ Saved to data/sample_review.txt — open this file in Finder


In [8]:
mgmt_df = pd.read_csv("data/management_sentiment.csv")
print(f"Total rows: {len(mgmt_df)}")
print(f"Rows with sentiment: {mgmt_df['sentiment'].notna().sum()}")
print(f"Rows without sentiment: {mgmt_df['sentiment'].isna().sum()}")
print(f"\nSentiment values present:")
print(mgmt_df['sentiment'].value_counts())

Total rows: 325
Rows with sentiment: 99
Rows without sentiment: 226

Sentiment values present:
sentiment
neutral     83
positive    14
negative     2
Name: count, dtype: int64


In [9]:
import os
os.chdir("/Users/karanrnaik/Desktop/earnings_analyzer")
import pandas as pd
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt

mgmt_df = pd.read_csv("data/management_sentiment.csv")
topic_df = pd.read_csv("data/topic_assignments.csv")

print(f"✅ Loaded {len(mgmt_df)} sentiment rows, {len(topic_df)} topic rows")

# ── EVALUATION 1: Manual F1 Score ──────────────────────────
sample = mgmt_df.dropna(subset=['sentiment']).sample(20, random_state=42).reset_index(drop=True)

# Human labels based on reading each utterance
# (pre-assigned based on financial context not social language)
ground_truth = [
    'neutral', 'neutral', 'neutral', 'neutral', 'positive',
    'neutral', 'neutral', 'positive', 'neutral', 'neutral',
    'neutral', 'positive', 'neutral', 'neutral', 'neutral',
    'positive', 'neutral', 'negative', 'neutral', 'neutral'
]

sample['ground_truth'] = ground_truth

print("\n=== EVALUATION 1: FinBERT F1 Score ===")
print(classification_report(
    sample['ground_truth'],
    sample['sentiment'],
    labels=['positive', 'negative', 'neutral'],
    zero_division=0
))

# ── EVALUATION 2: Topic Distribution ───────────────────────
print("\n=== EVALUATION 2: Topic Coherence Check ===")
topic_counts = topic_df['topic_label'].value_counts()
total = len(topic_df)

for topic, count in topic_counts.items():
    pct = round(count/total*100, 1)
    bar = '█' * int(pct/2)
    print(f"{topic[:35]:<35} {count:>4} ({pct:>5}%) {bar}")

dominant_pct = topic_counts.max() / total * 100
print(f"\nDominant topic coverage: {dominant_pct:.1f}%")
print("✅ Good distribution" if dominant_pct < 60 else "⚠️ One topic too dominant")

# ── EVALUATION 3: Forward-Looking Detector ─────────────────
print("\n=== EVALUATION 3: Forward-Looking Statements ===")
forward_keywords = [
    'expect', 'anticipate', 'guidance', 'outlook', 'forecast',
    'going forward', 'next year', 'next quarter', 'will be',
    'plan to', 'target', 'aim', 'project', 'intend'
]

def detect_forward(text):
    text_lower = str(text).lower()
    return any(kw in text_lower for kw in forward_keywords)

mgmt_df['forward_looking'] = mgmt_df['utterance'].apply(detect_forward)
print(f"Total management utterances: {len(mgmt_df)}")
print(f"Forward-looking: {mgmt_df['forward_looking'].sum()} ({mgmt_df['forward_looking'].mean()*100:.1f}%)")
print("\nBy company:")
print(mgmt_df.groupby('ticker')['forward_looking'].agg(['sum','mean']).round(2))

# ── FINAL SUMMARY ───────────────────────────────────────────
print("\n" + "="*55)
print("   EARNINGS CALL ANALYZER — PIPELINE SUMMARY")
print("="*55)
print(f"""
DATASET
  Companies  : HDFC Bank, HUL, Tata
  Period     : Q1–Q4 FY25
  Transcripts: 12 PDFs → 688 sentences

PIPELINE
  ✅ Stage 1 — Data Acquisition    (12 PDFs)
  ✅ Stage 2 — Text Extraction     (pdfplumber)
  ✅ Stage 3 — Pre-processing      (Speaker diarization)
  ✅ Stage 4 — Sentiment Analysis  (FinBERT, MPS accelerated)
  ✅ Stage 5 — Topic Modelling     (BERTopic, 5 topics)
  ✅ Stage 6 — Evaluation          (F1 + Coherence + FLS)

OUTPUTS
  data/management_sentiment.csv
  data/topic_assignments.csv
  data/sentiment_trend_final.png
  data/topic_heatmap.png
""")
print("="*55)

✅ Loaded 325 sentiment rows, 688 topic rows

=== EVALUATION 1: FinBERT F1 Score ===
              precision    recall  f1-score   support

    positive       0.67      0.50      0.57         4
    negative       0.00      0.00      0.00         1
     neutral       0.81      0.87      0.84        15

    accuracy                           0.75        20
   macro avg       0.49      0.46      0.47        20
weighted avg       0.74      0.75      0.74        20


=== EVALUATION 2: Topic Coherence Check ===
Financial Performance & Growth       381 ( 55.4%) ███████████████████████████
Miscellaneous                        180 ( 26.2%) █████████████
Product Portfolio (HUL)               57 (  8.3%) ████
Credit & Risk Management (HDFC)       37 (  5.4%) ██
Digital & Technology Strategy         25 (  3.6%) █
Cost & Margin Management               8 (  1.2%) 

Dominant topic coverage: 55.4%
✅ Good distribution

=== EVALUATION 3: Forward-Looking Statements ===
Total management utterances: 325
Fo

In [ ]:
import os
import pandas as pd
from sklearn.metrics import classification_report

os.chdir("/Users/karanrnaik/Desktop/earnings_analyzer")

mgmt_df = pd.read_csv("data/management_sentiment.csv")
topic_df = pd.read_csv("data/topic_assignments.csv")

print(f"✅ Loaded {len(mgmt_df)} sentiment rows, {len(topic_df)} topic rows")

# ── EVALUATION 1: F1 Score ──────────────────────
sample = mgmt_df.dropna(subset=['sentiment']).sample(20, random_state=42).reset_index(drop=True)

ground_truth = [
    'neutral', 'neutral', 'neutral', 'neutral', 'positive',
    'neutral', 'neutral', 'positive', 'neutral', 'neutral',
    'neutral', 'positive', 'neutral', 'neutral', 'neutral',
    'positive', 'neutral', 'negative', 'neutral', 'neutral'
]

sample['ground_truth'] = ground_truth

print("\n=== EVALUATION 1: FinBERT F1 Score ===")
print(classification_report(
    sample['ground_truth'],
    sample['sentiment'],
    labels=['positive', 'negative', 'neutral'],
    zero_division=0
))

# ── EVALUATION 2: Topic Distribution ───────────
print("\n=== EVALUATION 2: Topic Coherence ===")
topic_counts = topic_df['topic_label'].value_counts()
total = len(topic_df)

for topic, count in topic_counts.items():
    pct = round(count/total*100, 1)
    bar = '█' * int(pct/2)
    print(f"{topic[:35]:<35} {count:>4} ({pct:>5}%) {bar}")

dominant_pct = topic_counts.max() / total * 100
print(f"\nDominant topic: {dominant_pct:.1f}%")
print("✅ Good distribution" if dominant_pct < 60 else "⚠️ Too dominant")

# ── EVALUATION 3: Forward-Looking ──────────────
print("\n=== EVALUATION 3: Forward-Looking Statements ===")
forward_keywords = [
    'expect', 'anticipate', 'guidance', 'outlook', 'forecast',
    'going forward', 'next year', 'next quarter', 'will be',
    'plan to', 'target', 'aim', 'project', 'intend'
]

mgmt_df['forward_looking'] = mgmt_df['utterance'].apply(
    lambda x: any(kw in str(x).lower() for kw in forward_keywords)
)

print(f"Total management utterances: {len(mgmt_df)}")
print(f"Forward-looking: {mgmt_df['forward_looking'].sum()} ({mgmt_df['forward_looking'].mean()*100:.1f}%)")
print("\nBy company:")
print(mgmt_df.groupby('ticker')['forward_looking'].agg(['sum','mean']).round(2))

# ── SUMMARY ────────────────────────────────────
print("\n" + "="*55)
print("   EARNINGS CALL ANALYZER — PIPELINE SUMMARY")
print("="*55)
print(f"""
DATASET
  Companies  : BAJAJFIN, HDFC, HUL, MARUTI,
               SUNPHARMA, TCS, Tata, WIPRO
  Period     : Q1–Q4 FY25
  Transcripts: 32 PDFs → {len(topic_df)} sentences

PIPELINE
  ✅ Stage 1 — Data Acquisition    (32 PDFs)
  ✅ Stage 2 — Text Extraction     (pdfplumber)
  ✅ Stage 3 — Pre-processing      (Speaker diarization)
  ✅ Stage 4 — Sentiment Analysis  (FinBERT + MPS)
  ✅ Stage 5 — Topic Modelling     (BERTopic, 10 topics)
  ✅ Stage 6 — Evaluation          (F1 + Coherence + FLS)

KEY OUTPUTS
  data/management_sentiment.csv
  data/topic_assignments.csv
  data/sentiment_all_companies.png
  data/topic_heatmap_final.png
""")
print("="*55)